# PetMatch: treino v3 com mais fotos por raça (Stanford Dogs + Tsinghua Dogs)

**Por que esta versão:** o Stanford Dogs tem de 148 a 252 fotos por raça; depois de separar
validação e teste, sobravam só ~105 a 175 para treinar. Na avaliação do modelo v1, as raças
com "gêmeas" (Chihuahua 46%, Collie 33%, Husky 43%, Poodle Toy 42%) foram as que mais erraram.

**O que muda:**

- **Mais fotos:** junta o [Tsinghua Dogs](https://huggingface.co/datasets/giacomov/tsinghua_dogs)
  (licença CC-BY-4.0), com fotos do dia a dia e de vários ângulos. Cada raça passa a ter
  **pelo menos ~300 fotos de treino** (até `MAX_TSINGHUA` do Tsinghua + as do Stanford).
- **7 raças novas** que só existem no Tsinghua: Shiba Inu, Fila Brasileiro, Cão de Crista
  Chinês, Pastor Australiano, Cane Corso, Spitz Japonês e Bichon Frisé (total: **127 raças**).
- **Rede mais forte:** EfficientNetV2-B3 em 300×300, treinada em precisão mista (mais rápida na T4).
- **Detector de cachorro:** responde "não é um cachorro" em fotos sem cachorro.
- **Teste comparável:** o teste do Stanford é **o mesmo do v1** (semente 42), então dá para
  comparar direto com os 81,8% do modelo anterior. O Tsinghua tem um segundo teste, com fotos
  de outra fonte.

**Como usar:** *Ambiente de execução → Alterar o tipo → GPU T4*, depois *Executar tudo*.
Leva cerca de **1h30 a 2h** (o download dos dois datasets leva ~10 min).

No final você baixa três arquivos, que vão para `backend/modelo/`:
`modelo_racas.onnx`, `classes.json` e `modelo_cachorro.onnx`.

## 1. Conferir a GPU e ligar a precisão mista

In [ ]:
import numpy as np
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "Sem GPU: Ambiente de execução → Alterar o tipo → GPU T4"

# Contas em float16 na GPU (cerca de 2x mais rápido na T4); os pesos continuam em float32.
# Antes de exportar, o modelo é refeito em float32 (passo 16), então o ONNX fica igual ao de sempre.
keras.mixed_precision.set_global_policy("mixed_float16")

## 2. Conectar o Google Drive

O Colab apaga `/content` quando desconecta. O checkpoint e os arquivos finais vão para o Drive.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
PASTA_DRIVE = "/content/drive/MyDrive/pets_identificator_v3"
os.makedirs(PASTA_DRIVE, exist_ok=True)
print("Arquivos finais serão salvos em:", PASTA_DRIVE)

## 3. Nomes das raças em português

As 120 raças do Stanford (mesmos nomes do modelo atual, para o histórico continuar batendo)
e as 7 raças novas que vêm só do Tsinghua. Mude `INCLUIR_RACAS_NOVAS` para `False` se quiser
manter só as 120.

In [ ]:
INCLUIR_RACAS_NOVAS = True

# Final do nome da pasta do Stanford -> nome no app (na ordem dos códigos n02085620, n02085782...)
NOMES_PT = {
    "Chihuahua": "Chihuahua",
    "Japanese_spaniel": "Chin Japonês",
    "Maltese_dog": "Maltês",
    "Pekinese": "Pequinês",
    "Shih-Tzu": "Shih Tzu",
    "Blenheim_spaniel": "Cavalier King Charles Spaniel",
    "papillon": "Papillon",
    "toy_terrier": "Toy Terrier",
    "Rhodesian_ridgeback": "Rhodesian Ridgeback",
    "Afghan_hound": "Galgo Afegão",
    "basset": "Basset Hound",
    "beagle": "Beagle",
    "bloodhound": "Bloodhound",
    "bluetick": "Bluetick Coonhound",
    "black-and-tan_coonhound": "Black and Tan Coonhound",
    "Walker_hound": "Treeing Walker Coonhound",
    "English_foxhound": "Foxhound Inglês",
    "redbone": "Redbone Coonhound",
    "borzoi": "Borzói",
    "Irish_wolfhound": "Lébrel Irlandês",
    "Italian_greyhound": "Galguinho Italiano",
    "whippet": "Whippet",
    "Ibizan_hound": "Podengo Ibicenco",
    "Norwegian_elkhound": "Elkhound Norueguês",
    "otterhound": "Otterhound",
    "Saluki": "Saluki",
    "Scottish_deerhound": "Deerhound",
    "Weimaraner": "Weimaraner",
    "Staffordshire_bullterrier": "Staffordshire Bull Terrier",
    "American_Staffordshire_terrier": "American Staffordshire Terrier",
    "Bedlington_terrier": "Bedlington Terrier",
    "Border_terrier": "Border Terrier",
    "Kerry_blue_terrier": "Kerry Blue Terrier",
    "Irish_terrier": "Terrier Irlandês",
    "Norfolk_terrier": "Norfolk Terrier",
    "Norwich_terrier": "Norwich Terrier",
    "Yorkshire_terrier": "Yorkshire",
    "wire-haired_fox_terrier": "Fox Terrier de Pelo Duro",
    "Lakeland_terrier": "Lakeland Terrier",
    "Sealyham_terrier": "Sealyham Terrier",
    "Airedale": "Airedale Terrier",
    "cairn": "Cairn Terrier",
    "Australian_terrier": "Terrier Australiano",
    "Dandie_Dinmont": "Dandie Dinmont Terrier",
    "Boston_bull": "Boston Terrier",
    "miniature_schnauzer": "Schnauzer Miniatura",
    "giant_schnauzer": "Schnauzer Gigante",
    "standard_schnauzer": "Schnauzer Médio",
    "Scotch_terrier": "Terrier Escocês",
    "Tibetan_terrier": "Terrier Tibetano",
    "silky_terrier": "Silky Terrier",
    "soft-coated_wheaten_terrier": "Soft-Coated Wheaten Terrier",
    "West_Highland_white_terrier": "West Highland White Terrier",
    "Lhasa": "Lhasa Apso",
    "flat-coated_retriever": "Flat-Coated Retriever",
    "curly-coated_retriever": "Curly-Coated Retriever",
    "golden_retriever": "Golden Retriever",
    "Labrador_retriever": "Labrador",
    "Chesapeake_Bay_retriever": "Chesapeake Bay Retriever",
    "German_short-haired_pointer": "Braco Alemão de Pelo Curto",
    "vizsla": "Vizsla",
    "English_setter": "Setter Inglês",
    "Irish_setter": "Setter Irlandês",
    "Gordon_setter": "Gordon Setter",
    "Brittany_spaniel": "Spaniel Bretão",
    "clumber": "Clumber Spaniel",
    "English_springer": "Springer Spaniel Inglês",
    "Welsh_springer_spaniel": "Springer Spaniel Galês",
    "cocker_spaniel": "Cocker Spaniel",
    "Sussex_spaniel": "Sussex Spaniel",
    "Irish_water_spaniel": "Spaniel de Água Irlandês",
    "kuvasz": "Kuvasz",
    "schipperke": "Schipperke",
    "groenendael": "Pastor Belga Groenendael",
    "malinois": "Pastor Belga Malinois",
    "briard": "Briard",
    "kelpie": "Kelpie Australiano",
    "komondor": "Komondor",
    "Old_English_sheepdog": "Old English Sheepdog",
    "Shetland_sheepdog": "Pastor de Shetland",
    "collie": "Collie",
    "Border_collie": "Border Collie",
    "Bouvier_des_Flandres": "Bouvier de Flandres",
    "Rottweiler": "Rottweiler",
    "German_shepherd": "Pastor Alemão",
    "Doberman": "Dobermann",
    "miniature_pinscher": "Pinscher Miniatura",
    "Greater_Swiss_Mountain_dog": "Grande Boiadeiro Suíço",
    "Bernese_mountain_dog": "Boiadeiro Bernês",
    "Appenzeller": "Boiadeiro de Appenzell",
    "EntleBucher": "Boiadeiro de Entlebuch",
    "boxer": "Boxer",
    "bull_mastiff": "Bulmastife",
    "Tibetan_mastiff": "Mastim Tibetano",
    "French_bulldog": "Buldogue Francês",
    "Great_Dane": "Dogue Alemão",
    "Saint_Bernard": "São Bernardo",
    "Eskimo_dog": "Cão Esquimó",
    "malamute": "Malamute do Alasca",
    "Siberian_husky": "Husky Siberiano",
    "affenpinscher": "Affenpinscher",
    "basenji": "Basenji",
    "pug": "Pug",
    "Leonberg": "Leonberger",
    "Newfoundland": "Terra-Nova",
    "Great_Pyrenees": "Cão dos Pirineus",
    "Samoyed": "Samoieda",
    "Pomeranian": "Lulu da Pomerânia",
    "chow": "Chow Chow",
    "keeshond": "Keeshond",
    "Brabancon_griffon": "Petit Brabançon",
    "Pembroke": "Welsh Corgi Pembroke",
    "Cardigan": "Welsh Corgi Cardigan",
    "toy_poodle": "Poodle Toy",
    "miniature_poodle": "Poodle Miniatura",
    "standard_poodle": "Poodle Standard",
    "Mexican_hairless": "Pelado Mexicano",
    "dingo": "Dingo",
    "dhole": "Dhole",
    "African_hunting_dog": "Cão Selvagem Africano",
}

# Raças que só existem no Tsinghua Dogs
NOMES_NOVOS = {
    "Shiba_Dog": "Shiba Inu",
    "Fila_Braziliero": "Fila Brasileiro",
    "Chinese_Crested_Dog": "Cão de Crista Chinês",
    "Australian_Shepherd": "Pastor Australiano",
    "Cane_Carso": "Cane Corso",
    "Japanese_Spitzes": "Spitz Japonês",
    "Bichon_Frise": "Bichon Frisé",
}

import re

def chave(nome):
    """'curly-coated_retriever' e 'curly_coated_retriever' viram 'curlycoatedretriever'."""
    return re.sub(r"[^a-z]", "", nome.lower())

# O Tsinghua escreve "Brabancon_griffo" (sem o n final)
APELIDOS = {"brabancongriffo": "brabancongriffon"}
PT_POR_CHAVE = {chave(k): v for k, v in NOMES_PT.items()}
if INCLUIR_RACAS_NOVAS:
    PT_POR_CHAVE.update({chave(k): v for k, v in NOMES_NOVOS.items()})

todos = list(PT_POR_CHAVE.values())
assert len(set(todos)) == len(todos), "Dois nomes de raça iguais"
print(len(todos), "raças no total")

## 4. Baixar o Stanford Dogs (~750 MB, 120 raças)

Tenta o site de Stanford e, se estiver fora do ar, usa a cópia do Hugging Face (que vem com
as fotos numa pasta só; a célula recria as pastas por raça a partir do código de cada foto).

In [ ]:
import subprocess
from pathlib import Path

STANFORD = Path("/content/stanford/Images")

if not STANFORD.exists():
    ok = subprocess.run(
        "mkdir -p /content/stanford && wget -q --timeout=30 --tries=2 "
        "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar -O /content/images.tar "
        "&& tar -xf /content/images.tar -C /content/stanford",
        shell=True,
    ).returncode == 0
    if not ok:
        print("Site de Stanford fora do ar, baixando a cópia do Hugging Face…")
        from huggingface_hub import hf_hub_download
        arquivo = hf_hub_download("Alanox/stanford-dogs", "images.tar.gz", repo_type="dataset")
        subprocess.run(f"mkdir -p /content/stanford_plano && tar -xzf {arquivo} -C /content/stanford_plano", shell=True, check=True)
        fotos = sorted(Path("/content/stanford_plano").rglob("*.jpg"))
        codigos = sorted({f.name.split("_")[0] for f in fotos})
        assert len(codigos) == 120, f"Esperava 120 códigos de raça, achei {len(codigos)}"
        # NOMES_PT está na ordem dos códigos: o 1º código é o 1º nome, e assim por diante
        pasta_do_codigo = {c: STANFORD / f"{c}-{s}" for c, s in zip(codigos, NOMES_PT)}
        for p in pasta_do_codigo.values():
            p.mkdir(parents=True, exist_ok=True)
        for f in fotos:
            f.rename(pasta_do_codigo[f.name.split("_")[0]] / f.name)

pastas_stanford = sorted(p for p in STANFORD.iterdir() if p.is_dir())
print("Raças do Stanford:", len(pastas_stanford), "| exemplo:", pastas_stanford[0].name)
assert len(pastas_stanford) == 120

## 5. Baixar o Tsinghua Dogs (~2,7 GB, 130 raças)

Vem em arquivos Parquet com as fotos dentro. A célula salva cada foto na pasta da raça
correspondente e descarta arquivos corrompidos. As categorias "chinese_rural_dog", "teddy"
e "Black_sable" ficam de fora (não são raças definidas).

In [ ]:
import io
import json
import pyarrow.parquet as pq
from huggingface_hub import snapshot_download
from PIL import Image

TSINGHUA = Path("/content/tsinghua")
pasta_hf = Path(snapshot_download("giacomov/tsinghua_dogs", repo_type="dataset", allow_patterns="data/*.parquet"))
arquivos = sorted(pasta_hf.glob("data/*.parquet"))
print(len(arquivos), "arquivos Parquet")

# Nomes das categorias ficam nos metadados do Parquet
meta = json.loads(pq.read_schema(arquivos[0]).metadata[b"huggingface"])
nomes_tsinghua = meta["info"]["features"]["label"]["names"]

def raca_pt(nome_tsinghua):
    """'420-n000124-Chihuahua' -> 'Chihuahua' (ou None se a categoria fica de fora)."""
    k = chave(nome_tsinghua.split("-", 2)[2])
    return PT_POR_CHAVE.get(APELIDOS.get(k, k))

contagem = {"train": 0, "validation": 0, "descartadas": 0}
for arquivo in arquivos:
    parte = "validation" if "validation" in arquivo.name else "train"
    for lote in pq.ParquetFile(arquivo).iter_batches(batch_size=256, columns=["image", "label"]):
        for imagem, rotulo in zip(lote.column("image").to_pylist(), lote.column("label").to_pylist()):
            raca = raca_pt(nomes_tsinghua[rotulo])
            if raca is None:
                continue
            try:
                Image.open(io.BytesIO(imagem["bytes"])).verify()
            except Exception:
                contagem["descartadas"] += 1
                continue
            destino = TSINGHUA / parte / raca
            destino.mkdir(parents=True, exist_ok=True)
            (destino / f"t{contagem[parte]:06d}.jpg").write_bytes(imagem["bytes"])
            contagem[parte] += 1
    print(arquivo.name, contagem)

racas_tsinghua = sorted(p.name for p in (TSINGHUA / "train").iterdir())
print(len(racas_tsinghua), "raças do Tsinghua aproveitadas")

## 6. Montar treino, validação e teste

- **Stanford:** mesma divisão do v1 (70% treino / 15% validação / 15% teste, semente 42).
  O teste é idêntico ao do modelo anterior.
- **Tsinghua (treino):** até `MAX_TSINGHUA` fotos por raça vão para o treino. O limite evita que
  raças com milhares de fotos (Golden, Labrador) dominem o modelo e mantém o tempo de treino.
- **Tsinghua (validação oficial, ~40 fotos por raça):** vira um **segundo teste**, com fotos de
  outra fonte. Não entra no treino.

In [ ]:
import random
import shutil

MAX_TSINGHUA = 300
DESTINO = Path("/content/dataset")
if DESTINO.exists():
    shutil.rmtree(DESTINO)

def copiar(arquivos, parte, raca, prefixo=""):
    pasta = DESTINO / parte / raca
    pasta.mkdir(parents=True, exist_ok=True)
    for arquivo in arquivos:
        os.link(arquivo, pasta / f"{prefixo}{arquivo.name}")  # link: não ocupa espaço extra

random.seed(42)
for pasta in pastas_stanford:
    raca = NOMES_PT[pasta.name.split("-", 1)[1]]
    imagens = sorted(pasta.glob("*.jpg"))
    random.shuffle(imagens)
    n_treino, n_val = int(len(imagens) * 0.70), int(len(imagens) * 0.15)
    copiar(imagens[:n_treino], "train", raca, "s_")
    copiar(imagens[n_treino:n_treino + n_val], "val", raca, "s_")
    copiar(imagens[n_treino + n_val:], "test", raca, "s_")

sorteio = random.Random(7)
for raca in racas_tsinghua:
    imagens = sorted((TSINGHUA / "train" / raca).glob("*.jpg"))
    sorteio.shuffle(imagens)
    copiar(imagens[:MAX_TSINGHUA], "train", raca, "t_")
    validacao = TSINGHUA / "validation" / raca
    if validacao.exists():
        copiar(sorted(validacao.glob("*.jpg")), "test_tsinghua", raca)

for parte in ("train", "val", "test", "test_tsinghua"):
    racas = list((DESTINO / parte).iterdir())
    fotos = [len(list(r.iterdir())) for r in racas]
    print(f"{parte:14s} {len(racas):3d} raças, {sum(fotos):6d} fotos, por raça: mín {min(fotos)} / máx {max(fotos)}")

## 7. Carregar as imagens

A **ordem das classes** é a alfabética dos nomes das pastas, e é ela que vai para o
`classes.json`. O índice que o modelo devolve é a posição nessa lista: nunca reordene
o arquivo à mão.

In [ ]:
IMG = 300   # tamanho nativo da EfficientNetV2-B3
LOTE = 32
AUTOTUNE = tf.data.AUTOTUNE

def carregar(parte, embaralhar, classes=None):
    return keras.utils.image_dataset_from_directory(
        DESTINO / parte,
        image_size=(IMG, IMG),
        batch_size=LOTE,
        label_mode="int",
        class_names=classes,
        shuffle=embaralhar,
        seed=42,
    )

treino_ds = carregar("train", embaralhar=True)
CLASSES = treino_ds.class_names
NUM_CLASSES = len(CLASSES)
val_ds = carregar("val", embaralhar=False, classes=CLASSES)
teste_ds = carregar("test", embaralhar=False, classes=CLASSES)

# Raças novas não existem no Stanford: o teste do Tsinghua usa só as pastas que existem
racas_teste_t = sorted(p.name for p in (DESTINO / "test_tsinghua").iterdir())
teste_t_ds = carregar("test_tsinghua", embaralhar=False, classes=racas_teste_t)
print(NUM_CLASSES, "classes. Primeiras:", CLASSES[:5])

## 8. Aumento de dados (ângulos e enquadramentos diferentes)

Aplicado **só no treino** e **fora do modelo**. Espelhamento, rotação, zoom para longe e para
perto, deslocamento, contraste e brilho simulam fotos tiradas de outros ângulos e distâncias.

In [ ]:
aumento = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.08),
    keras.layers.RandomZoom(height_factor=(-0.2, 0.3), fill_mode="reflect"),
    keras.layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
    keras.layers.RandomContrast(0.2),
    keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="aumento")

treino_aug = (
    treino_ds
    .map(lambda x, y: (aumento(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
teste_ds = teste_ds.prefetch(AUTOTUNE)
teste_t_ds = teste_t_ds.prefetch(AUTOTUNE)

## 9. Montar o modelo (EfficientNetV2-B3)

A EfficientNetV2 do Keras já traz a normalização dentro dela e espera valores de 0 a 255,
o mesmo formato que a API envia. A última camada fica em float32 (exigência da precisão mista).

In [ ]:
def montar_modelo():
    # Nomes fixos: o passo 16 monta o modelo de novo e carrega os pesos pelo nome das camadas
    base = keras.applications.EfficientNetV2B3(
        input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet",
        include_preprocessing=True, name="base_efficientnet",
    )
    entrada = keras.Input(shape=(IMG, IMG, 3), name="imagem")
    x = base(entrada, training=False)
    x = keras.layers.GlobalAveragePooling2D(name="media_global")(x)
    x = keras.layers.Dropout(0.4, name="dropout")(x)
    saida = keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="probabilidades")(x)
    return keras.Model(entrada, saida, name="classificador_racas"), base

modelo, base = montar_modelo()
base.trainable = False  # fase 1: base congelada
modelo.summary(show_trainable=True)

## 10. Treino, fase 1 (só a camada nova)

O `ModelCheckpoint` salva no Drive a cada melhora: se o Colab cair, nada se perde
(veja "Se o Colab desconectar" no fim do notebook).

In [ ]:
parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
checkpoint = keras.callbacks.ModelCheckpoint(
    f"{PASTA_DRIVE}/checkpoint.weights.h5", monitor="val_loss", save_best_only=True, save_weights_only=True
)
metricas = ["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")]

modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=metricas)
hist1 = modelo.fit(treino_aug, validation_data=val_ds, epochs=5, callbacks=[parada, checkpoint])

## 11. Treino, fase 2 (ajuste fino)

Libera as últimas 80 camadas da base com taxa de aprendizado baixa. Compare a
`val_accuracy` com a do v1 (~83%).

In [ ]:
base.trainable = True
for camada in base.layers[:-80]:
    camada.trainable = False
for camada in base.layers:
    if isinstance(camada, keras.layers.BatchNormalization):
        camada.trainable = False  # normalização fica como no ImageNet

modelo.compile(optimizer=keras.optimizers.Adam(2e-5), loss="sparse_categorical_crossentropy", metrics=metricas)
hist2 = modelo.fit(treino_aug, validation_data=val_ds, epochs=6, callbacks=[parada, checkpoint])

## 12. Gráfico do treino (para o relatório)

In [ ]:
import matplotlib.pyplot as plt

def junta(chave_hist):
    return hist1.history[chave_hist] + hist2.history[chave_hist]

inicio_fase2 = len(hist1.history["accuracy"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(junta("accuracy"), label="treino"); ax1.plot(junta("val_accuracy"), label="validação")
ax1.set_title("Acurácia"); ax1.axvline(inicio_fase2 - 0.5, ls="--", c="gray")
ax2.plot(junta("loss"), label="treino"); ax2.plot(junta("val_loss"), label="validação")
ax2.set_title("Perda (loss)"); ax2.axvline(inicio_fase2 - 0.5, ls="--", c="gray")
for ax in (ax1, ax2):
    ax.set_xlabel("época"); ax.legend()
plt.savefig(f"{PASTA_DRIVE}/grafico_treino.png", dpi=120, bbox_inches="tight")
plt.show()

## 13. Avaliação: teste do Stanford (o mesmo do v1) e teste do Tsinghua

O teste do Stanford é comparável com o modelo atual (v1: **81,8%**). O do Tsinghua usa fotos
de outra fonte e dá uma ideia melhor de como o modelo vai em fotos do dia a dia.

> **Para o relatório:** o Stanford Dogs vem do ImageNet, usado no pré-treino da
> EfficientNetV2, então o número dele tende a ser otimista.

In [ ]:
from sklearn.metrics import classification_report

def avaliar(ds, nome, racas_do_ds):
    y_real = np.concatenate([y.numpy() for _, y in ds])
    # o rótulo do ds é a posição em racas_do_ds; converte para a posição em CLASSES
    y_real = np.array([CLASSES.index(racas_do_ds[i]) for i in y_real])
    prob = modelo.predict(ds, verbose=0)
    y_prev = prob.argmax(1)
    top3 = np.mean([r in p for r, p in zip(y_real, np.argsort(prob, 1)[:, -3:])])
    print(f"[{nome}] acerto: {(y_prev == y_real).mean():.1%}   top-3: {top3:.1%}   fotos: {len(y_real)}")
    return y_real, y_prev, prob

y_real, y_prev, prob_teste = avaliar(teste_ds, "teste Stanford", CLASSES)
y_real_t, y_prev_t, _ = avaliar(teste_t_ds, "teste Tsinghua", racas_teste_t)

presentes = sorted(set(y_real))
relatorio = classification_report(
    y_real, y_prev, labels=presentes, target_names=[CLASSES[i] for i in presentes], digits=3, output_dict=True, zero_division=0
)
with open(f"{PASTA_DRIVE}/relatorio_teste.txt", "w", encoding="utf-8") as f:
    f.write(classification_report(y_real, y_prev, labels=presentes, target_names=[CLASSES[i] for i in presentes], digits=3, zero_division=0))

print("\nRaças que mais erravam no v1 (acerto no v1 -> agora, teste Stanford):")
V1 = {"Chihuahua": 0.46, "Collie": 0.33, "Cão Esquimó": 0.35, "Poodle Toy": 0.42, "Husky Siberiano": 0.43,
      "Yorkshire": 0.50, "Whippet": 0.52, "Labrador": 0.85, "Golden Retriever": 0.87, "Pastor Alemão": 0.83}
for raca, antes in V1.items():
    print(f"  {raca:22s} {antes:4.0%} -> {relatorio[raca]['recall']:4.0%}")

piores = sorted((c for c in relatorio if c in CLASSES), key=lambda c: relatorio[c]["recall"])[:10]
print("\n10 raças com pior acerto agora:")
for c in piores:
    print(f"  {c:32s} {relatorio[c]['recall']:.0%}")

## 14. Escolher o limiar de confiança da API

Abaixo do limiar a API responde "Não reconhecido". A tabela usa o teste do Stanford.
Anote o valor sugerido: ele vai no `LIMIAR_CONFIANCA` do `backend/predictor.py`.

In [ ]:
confianca = prob_teste.max(1) * 100
acertou = y_prev == y_real
print(f"{'limiar':>7} {'cobertura':>10} {'acerto':>8}")
sugestao = None
for limiar in range(20, 80, 5):
    passa = confianca >= limiar
    precisao = acertou[passa].mean() if passa.any() else 0
    print(f"{limiar:>6}% {passa.mean():>10.1%} {precisao:>8.1%}")
    if sugestao is None and precisao >= 0.90:
        sugestao = limiar
print(f"\nLimiar sugerido: {sugestao}%  (primeiro com acerto >= 90%)")

## 15. Salvar pesos e classes no Drive

In [ ]:
modelo.save_weights(f"{PASTA_DRIVE}/modelo_racas.weights.h5")
with open(f"{PASTA_DRIVE}/classes.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, ensure_ascii=False, indent=2)
print(NUM_CLASSES, "classes salvas")

## 16. Refazer o modelo em float32 e exportar para ONNX

A precisão mista só serve para treinar mais rápido. Para a API, o modelo é montado de novo
em float32 com os mesmos pesos, e só então exportado.

In [ ]:
keras.mixed_precision.set_global_policy("float32")
modelo_final, _ = montar_modelo()
modelo_final.load_weights(f"{PASTA_DRIVE}/modelo_racas.weights.h5")

lote_teste = next(iter(teste_ds))[0].numpy().astype(np.float32)
diferenca = np.abs(modelo_final.predict(lote_teste, verbose=0) - modelo.predict(lote_teste, verbose=0)).max()
print(f"float32 x precisão mista: maior diferença {diferenca:.4f} (pequena é o esperado)")

modelo_final.export("/content/modelo_savedmodel")
!pip install -q tf2onnx onnxruntime
!python -m tf2onnx.convert --saved-model /content/modelo_savedmodel --output /content/modelo_racas.onnx --opset 13
!ls -lh /content/modelo_racas.onnx

## 17. Conferir se o ONNX dá o mesmo resultado que o Keras

In [ ]:
import onnxruntime as ort

sessao = ort.InferenceSession("/content/modelo_racas.onnx", providers=["CPUExecutionProvider"])
nome_entrada = sessao.get_inputs()[0].name
print("Entrada:", sessao.get_inputs()[0].shape, "| Saída:", sessao.get_outputs()[0].shape)

prob_keras = modelo_final.predict(lote_teste, verbose=0)
prob_onnx = sessao.run(None, {nome_entrada: lote_teste})[0]
diferenca = np.abs(prob_keras - prob_onnx).max()
iguais = (prob_keras.argmax(1) == prob_onnx.argmax(1)).mean()
print(f"Maior diferença: {diferenca:.6f}   previsões iguais: {iguais:.0%}")
assert prob_onnx.shape[1] == NUM_CLASSES
assert diferenca < 1e-3 and iguais == 1.0, "ONNX diferente do Keras!"
print("OK: o modelo ONNX é equivalente ao Keras.")

## 18. Detector de cachorro

O modelo de raças sempre escolhe uma raça, mesmo numa foto de gato ou de vaca. O detector é a
MobileNetV2 original do ImageNet: as categorias 151 a 268 do ImageNet são cachorros, e a soma
das probabilidades delas é a **probabilidade de ter um cachorro na foto**.

In [ ]:
detector_base = keras.applications.MobileNetV2(weights="imagenet", include_top=True)
entrada_det = keras.Input(shape=(224, 224, 3), name="imagem")
x = keras.layers.Rescaling(1 / 127.5, offset=-1)(entrada_det)  # 0..255 -> -1..1
probs_imagenet = detector_base(x, training=False)
prob_cachorro = keras.layers.Lambda(
    lambda p: keras.ops.sum(p[:, 151:269], axis=1, keepdims=True), name="prob_cachorro"
)(probs_imagenet)
detector = keras.Model(entrada_det, prob_cachorro, name="detector_cachorro")

pc = np.concatenate([
    detector.predict(tf.image.resize(x_lote, (224, 224)), verbose=0)[:, 0] for x_lote, _ in teste_ds
])
print(f"{'limiar':>7} {'cachorros aceitos':>18}")
for limiar in (0.2, 0.3, 0.4, 0.5):
    print(f"{limiar:>7.0%} {(pc >= limiar).mean():>18.1%}")
print("A API usa 30%: só bloqueia fotos em que o detector quase não vê cachorro.")

detector.export("/content/detector_savedmodel")
!python -m tf2onnx.convert --saved-model /content/detector_savedmodel --output /content/modelo_cachorro.onnx --opset 13
sessao_det = ort.InferenceSession("/content/modelo_cachorro.onnx", providers=["CPUExecutionProvider"])
lote_det = tf.image.resize(lote_teste, (224, 224)).numpy().astype(np.float32)
diferenca = np.abs(detector.predict(lote_det, verbose=0) - sessao_det.run(None, {sessao_det.get_inputs()[0].name: lote_det})[0]).max()
assert diferenca < 1e-3, "Detector ONNX diferente do Keras!"
print("OK: detector ONNX equivalente ao Keras.")

## 19. Copiar para o Drive e baixar

Depois de baixar, coloque os três arquivos em `backend/modelo/`:

1. `modelo_racas.onnx` e `classes.json` substituem os atuais (mesmos nomes).
2. `modelo_cachorro.onnx` é novo: a API passa a usar sozinha quando o arquivo existe.
3. No `backend/predictor.py`, troque `LIMIAR_CONFIANCA` pelo limiar sugerido no passo 14.

In [ ]:
import shutil
for arquivo in ("modelo_racas.onnx", "modelo_cachorro.onnx"):
    shutil.copy(f"/content/{arquivo}", f"{PASTA_DRIVE}/{arquivo}")
!ls -lh "{PASTA_DRIVE}"

from google.colab import files
files.download("/content/modelo_racas.onnx")
files.download(f"{PASTA_DRIVE}/classes.json")
files.download("/content/modelo_cachorro.onnx")

---

## Se o Colab desconectar no meio do treino

Rode as células 1 a 9 de novo (o download é refeito, ~10 min) e, no lugar do passo 10, rode:

```python
modelo.load_weights(f"{PASTA_DRIVE}/checkpoint.weights.h5")
```

Depois continue do passo 11 (ou do 13, se o treino já tinha terminado).